# VRP Research Expansion — the cube

Read-only visualization of the six result tables from migration 080. The shipped
harvest answered one cell of a 3-axis cube; this expansion corrects the measurement
layer (exact corp-action-adjusted forward RV + historical earnings calendar) and sweeps
three axes: **conditioning** (sector), **horizon** (T+5/20/60), **target** (harvest →
directional → ΔVRP-reversion).

Connects via `UW_SCAN_DB_*` env (defaults to the local dev DB). SELECT-only — never
mutates the DB.

In [1]:
import os
import psycopg
import pandas as pd
import matplotlib.pyplot as plt

dsn = (
    f"host={os.environ.get('UW_SCAN_DB_HOST', '127.0.0.1')} "
    f"dbname={os.environ.get('UW_SCAN_DB_NAME', 'option_wizard_local')} "
    f"user={os.environ.get('UW_SCAN_DB_USER', 'chenxi')}"
)
conn = psycopg.connect(dsn)

def tbl(name):
    return pd.read_sql(f'SELECT * FROM uw_scan.{name}', conn)

rv_val = tbl('vrp_rv_validation')
by_sector = tbl('vrp_harvest_by_sector')
multih = tbl('vrp_harvest_multihorizon')
directional = tbl('vrp_directional_verdicts')
dvrp = tbl('vrp_dvrp_reversion')
print({
    'rv_validation': len(rv_val), 'by_sector': len(by_sector),
    'multihorizon': len(multih), 'directional': len(directional), 'dvrp': len(dvrp),
})

Matplotlib is building the font cache; this may take a moment.


{'rv_validation': 342, 'by_sector': 51, 'multihorizon': 36, 'directional': 9, 'dvrp': 36}


/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/1500314272.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(f'SELECT * FROM uw_scan.{name}', conn)
/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/1500314272.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(f'SELECT * FROM uw_scan.{name}', conn)
/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/1500314272.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(f'SELEC

## Item 1 — is the trailing-21d RV approximation loose?
Mean |approx − exact| by horizon. Loosest at short horizons (few returns → noisy exact RV).

In [2]:
g = rv_val.groupby('horizon').agg(
    mean_abs_dev=('mean_abs_dev', 'mean'), corr=('corr', 'mean')
).reset_index()
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(g['horizon'].astype(str), g['mean_abs_dev'], color='#c0392b')
ax.set_xlabel('horizon (trading days)'); ax.set_ylabel('mean |approx − exact| (vol pts)')
ax.set_title('RV approximation looseness by horizon')
plt.tight_layout(); plt.show()
g

/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/1964560618.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


,horizon,mean_abs_dev,corr
0,5,0.184466,0.176090
1,20,0.135345,0.136867
2,60,0.117626,0.073195


## Item 4 — harvest decay by horizon (RICH buckets)
The premium peaks ~T+20 and decays by T+60 (single_name RICH drops out at T+60).

In [3]:
rich = multih[multih['deviation_class'] == 'RICH'].copy()
rich['mean_realized_vrp'] = rich['mean_realized_vrp'].astype(float)
fig, ax = plt.subplots(figsize=(7, 4))
for ac, sub in rich.groupby('asset_class'):
    sub = sub.sort_values('horizon')
    ax.plot(sub['horizon'], sub['mean_realized_vrp'], marker='o', label=ac)
ax.axhline(0.02, ls='--', c='gray', lw=0.8, label='sellable floor 0.02')
ax.set_xlabel('horizon'); ax.set_ylabel('mean realized VRP harvest')
ax.set_title('RICH harvest decay curve'); ax.legend()
plt.tight_layout(); plt.show()

/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/3385288411.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Item 2 — single-name harvest by sector (the WHERE)
Most sectors' RICH bucket IS sellable — single-name vol is sellable in specific sectors,
not uniformly. (Smaller per-sector n; treat as where-to-look, not blanket permission.)

In [4]:
sec = by_sector[by_sector['deviation_class'] == 'RICH'].copy()
sec['mean_realized_vrp'] = sec['mean_realized_vrp'].astype(float)
sec = sec.sort_values('mean_realized_vrp')
colors = ['#27ae60' if v == 'HARVEST_SELLABLE' else '#7f8c8d' for v in sec['verdict']]
fig, ax = plt.subplots(figsize=(7, max(4, 0.32 * len(sec))))
ax.barh(sec['sector'], sec['mean_realized_vrp'], color=colors)
ax.axvline(0.02, ls='--', c='gray', lw=0.8)
ax.set_xlabel('mean realized VRP harvest (RICH)'); ax.set_title('Single-name RICH harvest by sector')
plt.tight_layout(); plt.show()

/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/2119656270.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Item 5a — directional RICH−CHEAP differential
Single-name high-VRP names underperform low-VRP names at T+20/60 (BEARISH_TILT).

In [5]:
d = directional.copy()
d['mean_differential'] = d['mean_differential'].astype(float)
d['label'] = d['asset_class'] + ' T+' + d['horizon'].astype(str)
d = d.sort_values(['asset_class', 'horizon'])
colors = ['#27ae60' if x > 0 else '#c0392b' for x in d['mean_differential']]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(d['label'], d['mean_differential'], color=colors)
ax.axhline(0, c='black', lw=0.6)
ax.set_ylabel('RICH − CHEAP fwd return'); ax.set_title('Directional differential')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/1434604997.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


## Item 5b — ΔVRP reversion (RICH buckets)
Rich VRP reverts DOWN (negative forward ΔVRP), strengthening with horizon, strongest in single names.

In [6]:
r = dvrp[dvrp['deviation_class'] == 'RICH'].copy()
r['mean_fwd_dvrp'] = r['mean_fwd_dvrp'].astype(float)
piv = r.pivot_table(index='asset_class', columns='horizon', values='mean_fwd_dvrp')
ax = piv.plot(kind='bar', figsize=(8, 4))
ax.axhline(0, c='black', lw=0.6)
ax.set_ylabel('mean forward ΔVRP (RICH)'); ax.set_title('ΔVRP reversion — RICH reverts down')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
piv

/var/folders/p8/31vp59256fs2l6jg8gb79zz40000gn/T/ipykernel_74559/3465046331.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xticks(rotation=0); plt.tight_layout(); plt.show()


horizon,5,20,60
asset_class,,,
credit,-0.020373,-0.023159,-0.039061
index_macro,-0.013291,-0.039870,-0.060398
sector_etf,-0.017617,-0.052737,-0.062740
single_name,-0.016443,-0.086459,-0.104846
